# Balanced Self-Bootstrapped Identity Projection

This version fixes the collapse-to-anchor problem from the earlier self-bootstrap notebook.

Main changes:

1. The projector is trained against the `character_core_prompt`, not the full anchor prompt.
2. Bootstrap selection uses a balanced score:
   - identity similarity to the anchor
   - prompt-following similarity to the target edit prompt
3. Identity scale is reduced by default.
4. ControlNet pose strength is reduced by default so the image is not overly locked to the anchor layout.

This should keep the same character while improving editability.


In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux safetensors huggingface_hub


In [ ]:
from huggingface_hub import login

# Optional
login()


In [ ]:
from dataclasses import dataclass
from typing import List, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from controlnet_aux import OpenposeDetector
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, StableDiffusionPipeline
from IPython.display import display
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)


In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_openpose",
    torch_dtype=dtype,
).to(device)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "Lykon/DreamShaper",
    controlnet=controlnet,
    torch_dtype=dtype,
).to(device)

base_pipe = StableDiffusionPipeline.from_pretrained(
    "Lykon/DreamShaper",
    torch_dtype=dtype,
).to(device)

pipe.enable_attention_slicing()
base_pipe.enable_attention_slicing()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

pose_detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

print("Models loaded.")


In [ ]:
@dataclass
class CandidateResult:
    prompt: str
    seed: int
    identity_similarity: float
    prompt_similarity: float
    combined_score: float
    image: Image.Image


@dataclass
class CharacterProfile:
    character_core_prompt: str
    anchor_prompt: str
    anchor_seed: int
    anchor_image: Image.Image
    pose_image: Image.Image
    base_identity_embedding: torch.Tensor
    refined_identity_embedding: torch.Tensor
    identity_tokens: torch.Tensor
    projector_loss: float
    bootstrap_candidates: List[CandidateResult]


class MultiTokenIdentityProjector(nn.Module):
    def __init__(self, in_dim: int = 512, hidden_dim: int = 1024, num_tokens: int = 4, out_dim: int = 768):
        super().__init__()
        self.num_tokens = num_tokens
        self.out_dim = out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, num_tokens * out_dim),
        )

    def forward(self, identity_embedding: torch.Tensor) -> torch.Tensor:
        tokens = self.net(identity_embedding.float())
        return tokens.view(identity_embedding.shape[0], self.num_tokens, self.out_dim)


def build_generator(seed: int) -> torch.Generator:
    if device == "cuda":
        return torch.Generator(device="cuda").manual_seed(seed)
    return torch.Generator().manual_seed(seed)


def normalize_embedding(x: torch.Tensor) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)


def encode_sd_text(prompt: str, return_attention_mask: bool = False):
    text_input = pipe.tokenizer(
        prompt,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        text_embeddings = pipe.text_encoder(text_input.input_ids.to(device))[0]
    if return_attention_mask:
        return text_embeddings, text_input.attention_mask.to(device)
    return text_embeddings


def masked_mean(embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (embeddings.float() * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1.0)
    return summed / denom


def encode_clip_image(image: Image.Image) -> torch.Tensor:
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    with torch.no_grad():
        image_outputs = clip_model.vision_model(pixel_values=pixel_values)
        pooled = image_outputs.pooler_output
        image_features = clip_model.visual_projection(pooled)
    return normalize_embedding(image_features.float())


def encode_clip_text(text: str) -> torch.Tensor:
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    with torch.no_grad():
        text_outputs = clip_model.text_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = text_outputs.pooler_output
        text_features = clip_model.text_projection(pooled)
    return normalize_embedding(text_features.float())


def token_diversity_loss(identity_tokens: torch.Tensor) -> torch.Tensor:
    if identity_tokens.shape[1] <= 1:
        return torch.zeros(1, device=identity_tokens.device, dtype=identity_tokens.dtype).mean()

    norm_tokens = normalize_embedding(identity_tokens)
    sim = torch.matmul(norm_tokens, norm_tokens.transpose(1, 2))
    eye = torch.eye(sim.shape[-1], device=sim.device, dtype=torch.bool).unsqueeze(0)
    off_diag = sim.masked_select(~eye)
    return off_diag.mean()


def inject_identity_tokens(text_embeddings: torch.Tensor, identity_tokens: torch.Tensor, scale: float = 1.0) -> torch.Tensor:
    conditioned = text_embeddings.clone()
    num_tokens = identity_tokens.shape[1]
    conditioned[:, -num_tokens:, :] = conditioned[:, -num_tokens:, :] + (scale * identity_tokens.to(conditioned.dtype))
    return conditioned


def clear_negative_identity_slots(negative_embeddings: torch.Tensor, num_tokens: int) -> torch.Tensor:
    cleared = negative_embeddings.clone()
    cleared[:, -num_tokens:, :] = 0
    return cleared


def train_multi_token_projector(
    identity_embedding: torch.Tensor,
    character_core_prompt: str,
    num_identity_tokens: int = 4,
    hidden_dim: int = 1024,
    train_steps: int = 300,
    lr: float = 1e-4,
    diversity_weight: float = 0.05,
    norm_weight: float = 0.01,
) -> tuple[MultiTokenIdentityProjector, float]:
    projector = MultiTokenIdentityProjector(
        num_tokens=num_identity_tokens,
        hidden_dim=hidden_dim,
    ).to(device)
    projector.train()

    optimizer = torch.optim.Adam(projector.parameters(), lr=lr)
    target_embeddings, target_attention_mask = encode_sd_text(character_core_prompt, return_attention_mask=True)
    target_summary = normalize_embedding(masked_mean(target_embeddings, target_attention_mask).detach())
    identity_embedding = identity_embedding.float()

    last_loss = 0.0
    for step in range(train_steps):
        optimizer.zero_grad()

        identity_tokens = projector(identity_embedding)
        token_summary = normalize_embedding(identity_tokens.mean(dim=1))

        text_loss = 1 - F.cosine_similarity(token_summary, target_summary).mean()
        diversity_loss = token_diversity_loss(identity_tokens)
        token_norm_loss = (identity_tokens.norm(dim=-1).mean() - 1.0).abs()
        loss = text_loss + (diversity_weight * diversity_loss) + (norm_weight * token_norm_loss)

        loss.backward()
        optimizer.step()

        last_loss = float(loss.item())
        if step == 0 or (step + 1) % 25 == 0 or step == train_steps - 1:
            print(
                f"step {step + 1} | total {last_loss:.6f} | text {float(text_loss.item()):.6f} | diversity {float(diversity_loss.item()):.6f}"
            )

    projector.eval()
    return projector, last_loss


def generate_with_identity_tokens(
    identity_tokens: torch.Tensor,
    prompt: str,
    pose_image: Image.Image,
    seed: int = 1234,
    identity_scale: float = 1.0,
    generation_steps: int = 30,
    guidance_scale: float = 7.5,
    negative_prompt: str = "",
    controlnet_conditioning_scale: float = 0.55,
) -> Image.Image:
    text_embeddings = encode_sd_text(prompt)
    negative_embeddings = encode_sd_text(negative_prompt or "")

    conditioned_text = inject_identity_tokens(text_embeddings, identity_tokens, scale=identity_scale)
    conditioned_negative = clear_negative_identity_slots(negative_embeddings, identity_tokens.shape[1])

    image = pipe(
        prompt_embeds=conditioned_text,
        negative_prompt_embeds=conditioned_negative,
        image=pose_image,
        controlnet_conditioning_scale=controlnet_conditioning_scale,
        num_inference_steps=generation_steps,
        guidance_scale=guidance_scale,
        generator=build_generator(seed),
    ).images[0]
    return image


def candidate_score(
    image: Image.Image,
    anchor_identity: torch.Tensor,
    target_prompt: str,
    identity_weight: float = 0.65,
    prompt_weight: float = 0.35,
) -> tuple[float, float, float]:
    image_embedding = encode_clip_image(image)
    prompt_embedding = encode_clip_text(target_prompt)
    id_sim = float(F.cosine_similarity(image_embedding, anchor_identity).mean().item())
    prompt_sim = float(F.cosine_similarity(image_embedding, prompt_embedding).mean().item())
    score = (identity_weight * id_sim) + (prompt_weight * prompt_sim)
    return id_sim, prompt_sim, score


def show_grid(images: Sequence[Image.Image], cols: int = 2) -> Image.Image:
    if not images:
        raise ValueError("No images to show.")

    width, height = images[0].size
    rows = (len(images) + cols - 1) // cols
    grid = Image.new("RGB", (cols * width, rows * height), color=(255, 255, 255))

    for index, image in enumerate(images):
        x = (index % cols) * width
        y = (index // cols) * height
        grid.paste(image, (x, y))
    return grid


def create_character_balanced_bootstrap(
    anchor_prompt: str,
    character_core_prompt: str,
    bootstrap_prompts: Sequence[str],
    anchor_seed: int = 1234,
    anchor_steps: int = 30,
    anchor_guidance_scale: float = 8.0,
    train_steps: int = 300,
    projector_lr: float = 1e-4,
    num_identity_tokens: int = 4,
    bootstrap_seeds: Optional[Sequence[int]] = None,
    bootstrap_top_k: int = 2,
    bootstrap_identity_scale: float = 1.0,
    bootstrap_generation_steps: int = 20,
    bootstrap_guidance_scale: float = 7.5,
    bootstrap_controlnet_scale: float = 0.55,
    refine_anchor_weight: float = 0.55,
) -> CharacterProfile:
    print("Generating hidden anchor...")
    anchor_image = base_pipe(
        prompt=anchor_prompt,
        num_inference_steps=anchor_steps,
        guidance_scale=anchor_guidance_scale,
        generator=build_generator(anchor_seed),
    ).images[0]

    print("Extracting pose...")
    pose_image = pose_detector(anchor_image)

    print("Encoding anchor identity...")
    base_identity = encode_clip_image(anchor_image)

    print("Training projector on character core prompt...")
    projector, projector_loss = train_multi_token_projector(
        identity_embedding=base_identity,
        character_core_prompt=character_core_prompt,
        num_identity_tokens=num_identity_tokens,
        train_steps=train_steps,
        lr=projector_lr,
    )

    with torch.no_grad():
        bootstrap_tokens = projector(base_identity).to(dtype)

    if bootstrap_seeds is None:
        bootstrap_seeds = [1111, 2222, 3333, 4444]

    candidates: List[CandidateResult] = []
    candidate_embeddings: List[torch.Tensor] = []

    for index, prompt in enumerate(bootstrap_prompts):
        seed = int(bootstrap_seeds[index % len(bootstrap_seeds)])
        print(f"bootstrap candidate {index + 1}/{len(bootstrap_prompts)} | seed {seed}")
        image = generate_with_identity_tokens(
            identity_tokens=bootstrap_tokens,
            prompt=prompt,
            pose_image=pose_image,
            seed=seed,
            identity_scale=bootstrap_identity_scale,
            generation_steps=bootstrap_generation_steps,
            guidance_scale=bootstrap_guidance_scale,
            controlnet_conditioning_scale=bootstrap_controlnet_scale,
        )
        image_embedding = encode_clip_image(image)
        id_sim, prompt_sim, score = candidate_score(
            image=image,
            anchor_identity=base_identity,
            target_prompt=prompt,
        )
        print(f"identity={id_sim:.4f} | prompt={prompt_sim:.4f} | score={score:.4f}")
        candidates.append(
            CandidateResult(
                prompt=prompt,
                seed=seed,
                identity_similarity=id_sim,
                prompt_similarity=prompt_sim,
                combined_score=score,
                image=image,
            )
        )
        candidate_embeddings.append(image_embedding)

    ranked_indices = sorted(range(len(candidates)), key=lambda i: candidates[i].combined_score, reverse=True)
    selected_indices = ranked_indices[: min(bootstrap_top_k, len(ranked_indices))]
    selected_embeddings = torch.stack([candidate_embeddings[i] for i in selected_indices], dim=0)
    selected_mean = normalize_embedding(selected_embeddings.mean(dim=0))
    refined_identity = normalize_embedding(
        (refine_anchor_weight * base_identity) + ((1.0 - refine_anchor_weight) * selected_mean)
    )

    print("selected scores:", [round(candidates[i].combined_score, 4) for i in selected_indices])

    print("Retraining projector on refined identity...")
    final_projector, final_projector_loss = train_multi_token_projector(
        identity_embedding=refined_identity,
        character_core_prompt=character_core_prompt,
        num_identity_tokens=num_identity_tokens,
        train_steps=train_steps,
        lr=projector_lr,
    )

    with torch.no_grad():
        final_tokens = final_projector(refined_identity).to(dtype)

    return CharacterProfile(
        character_core_prompt=character_core_prompt,
        anchor_prompt=anchor_prompt,
        anchor_seed=anchor_seed,
        anchor_image=anchor_image,
        pose_image=pose_image,
        base_identity_embedding=base_identity,
        refined_identity_embedding=refined_identity,
        identity_tokens=final_tokens,
        projector_loss=final_projector_loss,
        bootstrap_candidates=candidates,
    )


def generate_batch(
    profile: CharacterProfile,
    prompt: str,
    seeds: Sequence[int],
    identity_scale: float = 1.0,
    generation_steps: int = 30,
    guidance_scale: float = 7.5,
    controlnet_conditioning_scale: float = 0.55,
) -> List[Image.Image]:
    images = []
    for seed in seeds:
        print(f"generating seed {seed}")
        image = generate_with_identity_tokens(
            identity_tokens=profile.identity_tokens,
            prompt=prompt,
            pose_image=profile.pose_image,
            seed=int(seed),
            identity_scale=identity_scale,
            generation_steps=generation_steps,
            guidance_scale=guidance_scale,
            controlnet_conditioning_scale=controlnet_conditioning_scale,
        )
        images.append(image)
    return images


In [ ]:
character_core_prompt = """
anime girl, long silver hair, violet eyes,
soft bangs, small mole under left eye,
detailed face, upper body portrait,
white background, high quality, sharp lineart
""".strip()

anchor_prompt = character_core_prompt + ", calm expression"

bootstrap_prompts = [
    character_core_prompt + ", surprised expression, raised eyebrows, open mouth",
    character_core_prompt + ", bright smile, cheerful expression",
    character_core_prompt + ", serious expression, looking slightly left",
    character_core_prompt + ", laughing expression, happy eyes",
]

profile = create_character_balanced_bootstrap(
    anchor_prompt=anchor_prompt,
    character_core_prompt=character_core_prompt,
    bootstrap_prompts=bootstrap_prompts,
    anchor_seed=1234,
    anchor_steps=30,
    anchor_guidance_scale=8.0,
    train_steps=250,
    projector_lr=1e-4,
    num_identity_tokens=4,
    bootstrap_seeds=[1111, 2222, 3333, 4444],
    bootstrap_top_k=2,
    bootstrap_identity_scale=1.0,
    bootstrap_generation_steps=20,
    bootstrap_guidance_scale=7.5,
    bootstrap_controlnet_scale=0.55,
    refine_anchor_weight=0.55,
)

print("final projector loss:", profile.projector_loss)
display(profile.anchor_image)
display(profile.pose_image)


In [ ]:
for item in profile.bootstrap_candidates:
    print(
        f"seed={item.seed} | id={item.identity_similarity:.4f} | prompt={item.prompt_similarity:.4f} | score={item.combined_score:.4f} | prompt={item.prompt}"
    )

display(show_grid([item.image for item in profile.bootstrap_candidates], cols=2))


In [ ]:
final_prompt = character_core_prompt + ", surprised expression, raised eyebrows, open mouth"
final_seeds = [5555, 6666, 7777, 8888]

final_images = generate_batch(
    profile=profile,
    prompt=final_prompt,
    seeds=final_seeds,
    identity_scale=1.0,
    generation_steps=30,
    guidance_scale=7.5,
    controlnet_conditioning_scale=0.55,
)

for image, seed in zip(final_images, final_seeds):
    id_sim, prompt_sim, score = candidate_score(image, profile.base_identity_embedding, final_prompt)
    print(f"seed={seed} | id={id_sim:.4f} | prompt={prompt_sim:.4f} | score={score:.4f}")

display(show_grid(final_images, cols=2))
